In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, accuracy_score)
import xgboost as xgb

from google.colab import files
uploaded = files.upload()


Saving Loan_default.csv to Loan_default.csv


In [ ]:

df = pd.read_csv('Loan_default.csv')
print(df.shape)
df.head()

(255347, 18)


,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [ ]:
print(df.info())
print(df.isnull().sum())
print(df['Default'].value_counts(normalize=True) * 100)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255347 entries, 0 to 255346
Data columns (total 18 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   LoanID          255347 non-null  object 
 1   Age             255347 non-null  int64  
 2   Income          255347 non-null  int64  
 3   LoanAmount      255347 non-null  int64  
 4   CreditScore     255347 non-null  int64  
 5   MonthsEmployed  255347 non-null  int64  
 6   NumCreditLines  255347 non-null  int64  
 7   InterestRate    255347 non-null  float64
 8   LoanTerm        255347 non-null  int64  
 9   DTIRatio        255347 non-null  float64
 10  Education       255347 non-null  object 
 11  EmploymentType  255347 non-null  object 
 12  MaritalStatus   255347 non-null  object 
 13  HasMortgage     255347 non-null  object 
 14  HasDependents   255347 non-null  object 
 15  LoanPurpose     255347 non-null  object 
 16  HasCoSigner     255347 non-null  object 
 17  Default   

In [ ]:
# Keep LoanID aside
loan_ids = df['LoanID']

cat_cols = ['Education', 'EmploymentType', 'MaritalStatus',
            'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner']

df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=False)
print(df_encoded.shape)
df_encoded.head()

(255347, 33)


,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,...,HasMortgage_Yes,HasDependents_No,HasDependents_Yes,LoanPurpose_Auto,LoanPurpose_Business,LoanPurpose_Education,LoanPurpose_Home,LoanPurpose_Other,HasCoSigner_No,HasCoSigner_Yes
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,...,True,False,True,False,False,False,False,True,False,True
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,...,False,True,False,False,False,False,False,True,False,True
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,...,True,False,True,True,False,False,False,False,True,False
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,...,False,True,False,False,True,False,False,False,True,False
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,...,False,False,True,True,False,False,False,False,True,False


In [ ]:
X = df_encoded.drop(['LoanID', 'Default'], axis=1)
y = df_encoded['Default']

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, loan_ids, test_size=0.2, random_state=42, stratify=y
)

num_cols = ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
            'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

In [ ]:
#Logistic Regression
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced')
log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)
y_prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression Results")
print(classification_report(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))

Logistic Regression Results
              precision    recall  f1-score   support

           0       0.94      0.67      0.79     45139
           1       0.22      0.70      0.33      5931

    accuracy                           0.68     51070
   macro avg       0.58      0.69      0.56     51070
weighted avg       0.86      0.68      0.73     51070

ROC-AUC: 0.753178399553392


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=2,
    verbose=1
)

start = time.time()
rf.fit(X_train, y_train)
print(f"Training time: {time.time() - start:.1f} seconds")

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest Results")
print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:   16.0s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:   31.7s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.


Training time: 31.8 seconds


[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.3s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:    1.1s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.3s


Random Forest Results
              precision    recall  f1-score   support

           0       0.94      0.74      0.83     45139
           1       0.24      0.63      0.35      5931

    accuracy                           0.73     51070
   macro avg       0.59      0.69      0.59     51070
weighted avg       0.86      0.73      0.77     51070

ROC-AUC: 0.7539360659503025


[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:    0.6s finished


In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    scale_pos_weight=(y_train.value_counts()[0] / y_train.value_counts()[1]),
    eval_metric='logloss', random_state=42
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost Results")
print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))

XGBoost Results
              precision    recall  f1-score   support

           0       0.94      0.71      0.81     45139
           1       0.23      0.66      0.34      5931

    accuracy                           0.71     51070
   macro avg       0.59      0.69      0.58     51070
weighted avg       0.86      0.71      0.76     51070

ROC-AUC: 0.7555297046095004


In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [accuracy_score(y_test, y_pred_lr),
                 accuracy_score(y_test, y_pred_rf),
                 accuracy_score(y_test, y_pred_xgb)],
    'ROC_AUC': [roc_auc_score(y_test, y_prob_lr),
                roc_auc_score(y_test, y_prob_rf),
                roc_auc_score(y_test, y_prob_xgb)]
})
print(results)
results.to_csv('comparison.csv', index=False)

                 Model  Accuracy   ROC_AUC
0  Logistic Regression  0.676385  0.753178
1        Random Forest  0.730018  0.753936
2              XGBoost  0.706990  0.755530


In [ ]:
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)

roc_data = pd.concat([
    pd.DataFrame({'Model': 'Logistic Regression', 'FPR': fpr_lr, 'TPR': tpr_lr}),
    pd.DataFrame({'Model': 'Random Forest', 'FPR': fpr_rf, 'TPR': tpr_rf}),
    pd.DataFrame({'Model': 'XGBoost', 'FPR': fpr_xgb, 'TPR': tpr_xgb})
])
roc_data.to_csv('roc_curve_data.csv', index=False)

In [ ]:
cm = confusion_matrix(y_test, y_pred_xgb)
cm_df = pd.DataFrame(cm, index=['Actual_NoDefault', 'Actual_Default'],
                      columns=['Predicted_NoDefault', 'Predicted_Default'])
cm_df.to_csv('confusion_matrix_xgb.csv')
print(cm_df)

                  Predicted_NoDefault  Predicted_Default
Actual_NoDefault                32175              12964
Actual_Default                   2000               3931


In [ ]:
importances = pd.Series(xgb_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False).reset_index()
importances.columns = ['Feature', 'Importance']
importances.to_csv('feature_importance_xgb.csv', index=False)
importances.head(15)

,Feature,Importance
0,Age,0.134269
1,InterestRate,0.071463
2,HasCoSigner_No,0.060124
3,Income,0.059191
4,EmploymentType_Full-time,0.058893
5,HasDependents_No,0.055827
6,MonthsEmployed,0.049065
7,LoanAmount,0.043952
8,EmploymentType_Unemployed,0.041023
9,HasMortgage_No,0.036664


In [ ]:
!pip install shap -q
import shap

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

mean_abs_shap = pd.DataFrame({
    'Feature': X_test.columns,
    'Mean_Abs_SHAP': np.abs(shap_values).mean(axis=0)
}).sort_values('Mean_Abs_SHAP', ascending=False)

mean_abs_shap.to_csv('shap_feature_importance.csv', index=False)
mean_abs_shap.head(15)

,Feature,Mean_Abs_SHAP
0,Age,0.521792
6,InterestRate,0.408099
4,MonthsEmployed,0.296290
1,Income,0.275862
2,LoanAmount,0.247295
29,HasCoSigner_No,0.124356
22,HasDependents_No,0.121072
3,CreditScore,0.113336
13,EmploymentType_Full-time,0.095194
5,NumCreditLines,0.093510


In [ ]:
!pip install shap -q
import shap

# Sample_size= 5000
sample_size = 5000
X_test_sample = X_test.sample(n=sample_size, random_state=42)
id_test_sample = id_test.loc[X_test_sample.index]
y_test_sample = y_test.loc[X_test_sample.index]
y_prob_xgb_sample = xgb_model.predict_proba(X_test_sample)[:, 1]

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_sample)

print("SHAP values computed. Shape:", shap_values.shape)

mean_abs_shap = pd.DataFrame({
    'Feature': X_test_sample.columns,
    'Mean_Abs_SHAP': np.abs(shap_values).mean(axis=0)
}).sort_values('Mean_Abs_SHAP', ascending=False)

mean_abs_shap.to_csv('shap_feature_importance.csv', index=False)
mean_abs_shap.head(15)

SHAP values computed. Shape: (5000, 31)


,Feature,Mean_Abs_SHAP
0,Age,0.521086
6,InterestRate,0.406693
4,MonthsEmployed,0.296032
1,Income,0.275549
2,LoanAmount,0.249119
29,HasCoSigner_No,0.124819
22,HasDependents_No,0.121776
3,CreditScore,0.113229
13,EmploymentType_Full-time,0.094985
5,NumCreditLines,0.093574


In [ ]:
shap_df = pd.DataFrame(shap_values, columns=X_test_sample.columns)
shap_df['LoanID'] = id_test_sample.values
shap_df['ActualDefault'] = y_test_sample.values
shap_df['PredictedProbability'] = y_prob_xgb_sample

cols = ['LoanID', 'ActualDefault', 'PredictedProbability'] + list(X_test_sample.columns)
shap_df = shap_df[cols]
shap_df.to_csv('shap_values_by_loan.csv', index=False)
print(shap_df.shape)
shap_df.head()

(5000, 34)


,LoanID,ActualDefault,PredictedProbability,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,...,HasMortgage_Yes,HasDependents_No,HasDependents_Yes,LoanPurpose_Auto,LoanPurpose_Business,LoanPurpose_Education,LoanPurpose_Home,LoanPurpose_Other,HasCoSigner_No,HasCoSigner_Yes
0,JBSLEMOBFK,0,0.572236,0.109123,-0.236471,-0.221978,0.080562,0.415961,-0.036754,-0.507049,...,0.0,0.141116,0.0,0.003075,-0.009042,0.000571,0.036882,0.002479,0.222348,0.0
1,I2Q15A1VQV,0,0.306194,-0.986066,0.498751,-0.598663,0.013269,0.267879,0.045373,0.259320,...,0.0,-0.120983,0.0,-0.004809,-0.028868,0.001178,0.030188,0.002727,-0.074328,0.0
2,55OUB1X4TE,0,0.163425,-0.248306,-0.184638,-0.216621,0.153059,-0.556099,-0.084982,-0.343526,...,0.0,-0.148328,0.0,-0.007867,-0.005447,0.000162,0.056714,-0.000375,0.140538,0.0
3,E7SBV6XE8A,1,0.631470,0.655158,0.119625,-0.482730,-0.251494,0.428930,-0.078407,0.478853,...,0.0,-0.169532,0.0,0.001007,-0.004799,0.000032,0.053635,-0.016453,0.115968,0.0
4,MLDNQ4L598,1,0.256042,-0.669060,0.067757,0.014166,0.032108,0.305690,-0.120458,-0.686106,...,0.0,0.091514,0.0,0.003471,-0.011940,-0.000426,0.036307,0.000612,-0.156680,0.0


In [ ]:
scored_df = df.loc[X_test.index].copy()
scored_df['PredictedProbability'] = y_prob_xgb
scored_df['PredictedDefault'] = y_pred_xgb
scored_df['ActualDefault'] = y_test.values

scored_df['RiskBand'] = pd.cut(
    scored_df['PredictedProbability'],
    bins=[0, 0.25, 0.5, 0.75, 1.0],
    labels=['Low', 'Medium', 'High', 'Very High']
)

scored_df.to_csv('scored_loans.csv', index=False)
scored_df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,...,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default,PredictedProbability,PredictedDefault,ActualDefault,RiskBand
211648,SJJNN2DFZ7,54,20956,87156,789,102,4,8.99,24,0.65,...,Single,Yes,Yes,Education,No,0,0.212627,0,0,Low
201101,GOC4OI55T8,49,138499,227279,603,102,4,2.46,12,0.67,...,Married,Yes,No,Home,No,0,0.051174,0,0,Low
140423,QO6XNPEH8P,69,22538,48532,337,48,2,16.15,24,0.74,...,Divorced,Yes,Yes,Other,No,0,0.303914,0,0,Medium
204530,3VLC51ZYU8,58,34754,134218,531,107,3,24.73,60,0.55,...,Married,No,No,Education,No,0,0.479172,0,0,Medium
166481,OUGXQD95MI,35,47430,130554,484,30,2,2.19,48,0.80,...,Married,No,No,Home,Yes,0,0.340461,0,0,Medium


In [ ]:
from google.colab import files

for f in ['comparison.csv', 'roc_curve_data.csv', 'confusion_matrix_xgb.csv',
          'feature_importance_xgb.csv', 'shap_feature_importance.csv',
          'shap_values_by_loan.csv', 'scored_loans.csv']:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (4 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>